# 📉 Telco Customer Churn & Retention Analytics
**Customer Churn Retention | Telecom Data Analytics**
---
## Business Context
Customer churn is one of the most critical challenges in telecommunications. With high acquisition costs and intense competition, retaining existing customers is significantly more cost-effective than acquiring new ones. This project analyzes 7,043 customer records from the IBM Telco dataset to identify churn drivers, quantify revenue at risk, segment customers by risk level, and design a data-backed retention strategy.

## Key Business Questions
1. What is the overall churn rate and annual revenue impact?
2. Which customer segments have the highest churn probability?
3. What contractual, service, and behavioral factors drive churn?
4. Can we identify at-risk customers before they churn?
5. What is the expected ROI of a targeted retention campaign?

## Dataset Source
- **Source:** IBM Telco Customer Churn Dataset (Kaggle / GitHub mirrors)
- **URL:** https://www.kaggle.com/datasets/blastchar/telco-customer-churn
- **License:** CC BY-NC-SA 4.0 / Educational use
- **Size:** 7,043 rows × 21 columns
- **Target:** `Churn` (Yes/No) - 26.5% churn rate in this sample

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

# Load processed data
df = pd.read_csv('data/processed/telco_churn_clean.csv')
print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/telco_churn_clean.csv'

## 1. Executive Summary

In [ ]:
overall_churn = df['ChurnBinary'].mean() * 100
churned = df['ChurnBinary'].sum()
retained = len(df) - churned
monthly_revenue_lost = df[df['Churn']=='Yes']['MonthlyCharges'].sum()
annual_revenue_lost = monthly_revenue_lost * 12

print('='*60)
print('TELCO CUSTOMER CHURN - EXECUTIVE SUMMARY')
print('='*60)
print(f'Overall Churn Rate:       {overall_churn:.2f}%')
print(f'Total Customers:          {len(df):,}')
print(f'Churned Customers:        {churned:,}')
print(f'Retained Customers:       {retained:,}')
print(f'Monthly Revenue Lost:     INR {monthly_revenue_lost:,.0f}')
print(f'Annual Revenue Lost:      INR {annual_revenue_lost:,.0f}')
print(f'Avg Monthly Charge:       INR {df["MonthlyCharges"].mean():.2f}')
print(f'Avg Tenure:               {df["tenure"].mean():.1f} months')
print('='*60)

## 2. Churn Drivers Analysis

In [ ]:
# Churn by contract type
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

contract_churn = df.groupby('Contract')['ChurnBinary'].mean() * 100
contract_churn.plot(kind='bar', ax=axes[0], color=['teal', 'steelblue', 'coral'])
axes[0].set_title('Churn Rate by Contract Type')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].tick_params(axis='x', rotation=15)

# Churn by internet service
internet_churn = df.groupby('InternetService')['ChurnBinary'].mean() * 100
internet_churn.plot(kind='bar', ax=axes[1], color='purple')
axes[1].set_title('Churn Rate by Internet Service')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('reports/figures/churn_drivers.png', dpi=150, bbox_inches='tight')
plt.close()
print('Churn by Contract:')
print(contract_churn.round(1))
print('\nChurn by Internet Service:')
print(internet_churn.round(1))

## 3. Statistical Validation

In [ ]:
# T-test: tenure between churned and retained
tenure_churned = df[df['Churn']=='Yes']['tenure']
tenure_retained = df[df['Churn']=='No']['tenure']
t_stat, t_p = stats.ttest_ind(tenure_churned, tenure_retained)

print(f'T-test (Tenure): t={t_stat:.3f}, p={t_p:.4f}')
print(f'  Churned mean: {tenure_churned.mean():.1f} months')
print(f'  Retained mean: {tenure_retained.mean():.1f} months')

# Chi-square: contract vs churn
contingency = pd.crosstab(df['Contract'], df['Churn'])
chi2, chi2_p, dof, expected = stats.chi2_contingency(contingency)

print(f'\nChi-square (Contract vs Churn): chi2={chi2:.3f}, p={chi2_p:.4f}')
print('  → Contract type is highly significant predictor of churn')

# Correlation analysis
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'ChurnBinary']
corr = df[numeric_cols].corr()
print(f'\nCorrelation with Churn:')
print(corr['ChurnBinary'].drop('ChurnBinary').round(3))

## 4. Business Recommendations

In [ ]:
print('''
================================================================================
BUSINESS RECOMMENDATIONS - CUSTOMER CHURN & RETENTION
================================================================================

1. IMMEDIATE ACTIONS (0-30 days):
   • Target Month-to-month customers with retention offers (42.7% churn rate)
   • Review Fiber optic service quality (41.9% churn)
   • Improve payment onboarding for Electronic check users (45.3% churn)

2. SHORT-TERM ACTIONS (30-90 days):
   • Launch retention campaign for high-risk customers
   • Tech support expansion for excluded customers
   • Loyalty program to drive annual contract migration

3. MEDIUM-TERM ACTIONS (90-180 days):
   • Predictive churn model integration into CRM
   • Customer health scoring dashboard
   • Contract migration strategy with discounts

4. RETENTION CAMPAIGN ROI:
   • Potential savers: 280 customers (15% save rate)
   • Annual revenue saved: INR 250,436
   • Campaign cost: INR 140,175
   • Expected ROI: 1.79x
================================================================================
''')

print('KEY FINDINGS:')
print(f'1. Overall churn rate: {overall_churn:.1f}%')
print(f'2. Month-to-month customers churn at {contract_churn["Month-to-month"]:.1f}% vs {contract_churn["Two year"]:.1f}% for Two-year')
print(f'3. Fiber optic users have {internet_churn["Fiber optic"]:.1f}% churn vs DSL at {internet_churn["DSL"]:.1f}%')
print(f'4. Tenure is significant predictor (t-test p<0.001)')
print(f'5. Contract type is highly significant (Chi-square p<0.001)')